# Activ Fitness Finder - Dokumentation

Persoenliche PWA (Progressive Web App) um alle Activ Fitness Studios der **Kategorie 1 und 2** auf einer interaktiven Karte anzuzeigen.

## Projektstruktur

```
Activ/
  index.html          # Haupt-App (PWA mit Leaflet-Karte)
  studios.json         # Alle 97 Studios (Kat 1 + Kat 2) mit Koordinaten
  manifest.json        # PWA Manifest fuer Homescreen-Installation
  sw.js                # Service Worker fuer Offline-Caching
  icon-192.png         # App-Icon 192x192
  icon-512.png         # App-Icon 512x512
  dokumentation.ipynb  # Dieses Notebook
  activ_fitness_kategorie2.json  # Rohdaten nur Kat 2
```

## 1. Datenerhebung

Alle Studiodaten wurden direkt von **activfitness.ch** gescraped:

1. Hauptseite `activfitness.ch/studios/` fuer die Studioliste und Kategorie-Zuordnung
2. Jede einzelne Studio-Detailseite fuer exakte GPS-Koordinaten, Adressen und Telefonnummern
3. Die Koordinaten stammen aus dem schema.org JSON-LD Markup der Seiten

### Abo-Logik
- **Kategorie 2 Abo** = Zugang zu **Kat 1 + Kat 2** Studios
- Kat 3 und Swiss Fit sind NICHT enthalten

In [ ]:
import json
import pandas as pd

# Studiodaten laden
with open('studios.json', 'r') as f:
    studios = json.load(f)

df = pd.DataFrame(studios)
print(f"Gesamtanzahl Studios: {len(df)}")
print(f"\nAufteilung nach Kategorie:")
print(df['kategorie'].value_counts().to_string())
print(f"\nStaedte mit den meisten Studios:")
print(df['city'].value_counts().head(10).to_string())

In [ ]:
# Alle Studios als Tabelle
df[['name', 'address', 'zip', 'city', 'kategorie']].head(20)

## 2. Visualisierung der Standorte

In [ ]:
try:
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(figsize=(10, 12))
    
    kat1 = df[df['kategorie'] == 1]
    kat2 = df[df['kategorie'] == 2]
    
    ax.scatter(kat2['lng'], kat2['lat'], c='#e63946', s=30, alpha=0.7, label=f'Kategorie 2 ({len(kat2)})', zorder=2)
    ax.scatter(kat1['lng'], kat1['lat'], c='#ffd700', s=80, marker='*', label=f'Kategorie 1 ({len(kat1)})', zorder=3)
    
    ax.set_xlabel('Laengengrad')
    ax.set_ylabel('Breitengrad')
    ax.set_title('Activ Fitness Studios - Kat 1 + Kat 2 Abo', fontsize=14, fontweight='bold')
    ax.legend()
    ax.set_facecolor('#1a1a2e')
    fig.patch.set_facecolor('#1a1a2e')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    ax.title.set_color('white')
    ax.legend(facecolor='#16213e', edgecolor='white', labelcolor='white')
    
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib nicht installiert - pip install matplotlib')

## 3. Distanzberechnung

Die App verwendet die **Haversine-Formel** um die Luftlinie zwischen dem Benutzerstandort und jedem Studio zu berechnen.

In [ ]:
import math

def haversine(lat1, lng1, lat2, lng2):
    """Distanz in km zwischen zwei GPS-Koordinaten (Luftlinie)."""
    R = 6371  # Erdradius in km
    dlat = math.radians(lat2 - lat1)
    dlng = math.radians(lng2 - lng1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlng/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

# Beispiel: Distanz von Zuerich HB zu allen Studios
my_lat, my_lng = 47.3769, 8.5417  # Zuerich HB

df['distanz_km'] = df.apply(lambda r: round(haversine(my_lat, my_lng, r['lat'], r['lng']), 1), axis=1)
df_sorted = df.sort_values('distanz_km')

print(f"Die 10 naechsten Studios von Zuerich HB:\n")
for i, row in df_sorted.head(10).iterrows():
    print(f"  {row['distanz_km']:5.1f} km  -  {row['name']} ({row['address']}, {row['city']})")

## 4. App

**Live:** https://kitty365.github.io/activ-finder/

### Auf iPhone installieren (PWA)
1. URL in Safari oeffnen
2. Teilen-Button > **Zum Home-Bildschirm**
3. Fertig

### Features
- Interaktive Karte mit allen 97 Studios (OpenStreetMap/Leaflet)
- GPS-Standortermittlung
- 5 naechste Studios direkt sichtbar
- Suchfunktion mit Geocoding (beliebiger Ort in der Schweiz)
- Navigation per Apple Maps
- Direktanruf per Tap
- Kategorie-Badges
- Offline-faehig (Service Worker)